In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession, Row
from delta import *
from pyspark.sql.types import (LongType, StringType, StructField, StructType,BooleanType, ArrayType, IntegerType, DoubleType)
from pyspark.sql.functions import (col, trim, when, greatest, expr)

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/demo'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/Data-Table.csv"
# se tiver outro nome no HDFS, ajusta aqui

df_bronze = (
    spark.read
        .option("header", "true")     # tem cabeçalho
        .option("delimiter", ",")     # CSV normal
        .csv(hdfs_path)
)

df_bronze.printSchema()
df_bronze.show(5)

root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Indicator Name: string (nullable = true)
 |-- 2003: string (nullable = true)
 |-- 2004: string (nullable = true)
 |-- 2005: string (nullable = true)
 |-- 2006: string (nullable = true)
 |-- 2007: string (nullable = true)
 |-- 2008: string (nullable = true)
 |-- 2009: string (nullable = true)
 |-- 2010: string (nullable = true)
 |-- 2011: string (nullable = true)
 |-- 2012: string (nullable = true)

+------------+------------+--------------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|Country Name|Country Code|      Indicator Name|   2003|   2004|   2005|   2006|   2007|   2008|   2009|   2010|   2011|   2012|
+------------+------------+--------------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|       Aruba|         ABW|International tou...| 642000| 728000| 733000| 694000| 772000| 827000| 813000| 824000| 

In [4]:
anos = [c for c in df_bronze.columns if c.isdigit()]
print("Colunas de anos:", anos)

df_silver = (
    df_bronze
        .withColumn("country_name",  trim(col("Country Name"))) #remover espaços
        .withColumn("country_code",  trim(col("Country Code")))
        .withColumn("indicator_name", trim(col("Indicator Name")))
)

for a in anos:
    df_silver = (
        df_silver
            .withColumn(a, col(a).cast("double"))
            .withColumn(a, col(a).cast(LongType()))   #evita notação científica
            .withColumn(a, when(col(a) < 0, None).otherwise(col(a)))
    )
df_silver = df_silver.filter(greatest(*[col(a) for a in anos]).isNotNull())

df_silver.printSchema()
expr_stack = "stack({n}, {pairs}) as (year, arrivals)".format(
    n=len(anos),
    pairs=", ".join([f"'{a}', `{a}`" for a in anos])
)

df_silver_long = (
    df_silver
        .select("country_name", "country_code", "indicator_name", expr(expr_stack))
        .where(col("arrivals").isNotNull())
        .withColumn("year", col("year").cast("int"))
        .withColumn("arrivals", col("arrivals").cast(LongType()))  #garantir inteiro, sem E
)

df_silver_long = df_silver_long.drop("indicator_name")

df_silver_long.printSchema()
df_silver_long.show(20, truncate=False)

Colunas de anos: ['2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012']
### Silver (wide) – Schema ###
root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Indicator Name: string (nullable = true)
 |-- 2003: long (nullable = true)
 |-- 2004: long (nullable = true)
 |-- 2005: long (nullable = true)
 |-- 2006: long (nullable = true)
 |-- 2007: long (nullable = true)
 |-- 2008: long (nullable = true)
 |-- 2009: long (nullable = true)
 |-- 2010: long (nullable = true)
 |-- 2011: long (nullable = true)
 |-- 2012: long (nullable = true)
 |-- country_name: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- indicator_name: string (nullable = true)

root
 |-- country_name: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- arrivals: long (nullable = true)

+------------+------------+----+--------+
|country_name|country_code|year|arrivals|
+---

In [5]:
delta_path_silver = "hdfs://hdfs-nn:9000/demo/silver/tourism_silver/"

(
    df_silver_long
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("year")
        .save(delta_path_silver)
)  #guardo os dados em HDFS (formato Delta)

In [6]:
spark.sql(
    """
    SHOW DATABASES
    """
).show()

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [7]:
spark.sql(
    """
    DROP TABLE IF EXISTS demo.tourism_silver
    """
)   #registar a tabela no Hive, que aponta para esses mesmos dados

DataFrame[]

In [8]:
spark.sql(
    """
    CREATE EXTERNAL TABLE IF NOT EXISTS demo.tourism_silver (
        country_name   STRING,
        country_code   STRING,
        year           INT,
        arrivals       BIGINT
    )
    USING DELTA
    PARTITIONED BY (year)
    LOCATION 'hdfs://hdfs-nn:9000/demo/silver/tourism_silver'
    """
)

DataFrame[]

In [9]:
spark.sql("SHOW TABLES FROM demo").show()

spark.sql(
    """
    SELECT 
        country_name,
        country_code,
        year,
        arrivals
    FROM demo.tourism_silver
    ORDER BY country_name, year
    """
).show(20, truncate=False)

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|     demo|boxoffice_silver|      false|
|     demo| deltalake_table|      false|
|     demo|  tourism_silver|      false|
+---------+----------------+-----------+

+------------+------------+----+--------+
|country_name|country_code|year|arrivals|
+------------+------------+----+--------+
|Albania     |ALB         |2003|557000  |
|Albania     |ALB         |2004|645000  |
|Albania     |ALB         |2005|748000  |
|Albania     |ALB         |2006|937000  |
|Albania     |ALB         |2007|1127000 |
|Albania     |ALB         |2008|1420000 |
|Albania     |ALB         |2009|1856000 |
|Albania     |ALB         |2010|2417000 |
|Albania     |ALB         |2011|2932000 |
|Albania     |ALB         |2012|3514000 |
|Algeria     |DZA         |2003|1166000 |
|Algeria     |DZA         |2004|1234000 |
|Algeria     |DZA         |2005|1443000 |
|Algeria     |DZA         |2006|1638000 

In [10]:
spark.sql("SELECT * FROM demo.tourism_silver").toPandas()

,country_name,country_code,year,arrivals
0,Aruba,ABW,2012,904000
1,Angola,AGO,2012,528000
2,Albania,ALB,2012,3514000
3,Argentina,ARG,2012,5585000
4,Armenia,ARM,2012,843000
...,...,...,...,...
1524,Samoa,WSM,2004,98000
1525,"Yemen, Rep.",YEM,2004,274000
1526,South Africa,ZAF,2004,6678000
1527,Zambia,ZMB,2004,515000


In [11]:
spark.stop()